In [1]:
!pip install -q "openai-agents[litellm]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.5/968.5 kB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.8 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("done")

done


In [3]:
from agents import Agent, Runner, set_tracing_disabled
from agents.extensions.models.litellm_model import LitellmModel

In [62]:
import agents.memory
print(dir(agents.memory))

['Any', 'OpenAIConversationsSession', 'OpenAIResponsesCompactionArgs', 'OpenAIResponsesCompactionAwareSession', 'OpenAIResponsesCompactionSession', 'Session', 'SessionABC', 'SessionInputCallback', 'SessionSettings', 'TYPE_CHECKING', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__getattr__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'annotations', 'is_openai_responses_compaction_aware_session', 'openai_conversations_session', 'openai_responses_compaction_session', 'session', 'session_settings', 'util']


In [4]:
set_tracing_disabled(True)


In [5]:
gemini_model = LitellmModel(
    model="gemini/gemini-2.5-flash",
    api_key=os.environ["GEMINI_API_KEY"]
)

In [6]:
test_agent = Agent(
    name="Test Agent",
    instructions="You are a helpful software engineering assistant.",
    model=gemini_model
)

result = await Runner.run(
    test_agent,
    "What is FastAPI? Answer in two sentences."
)

print(result.final_output)

FastAPI is a modern, fast (high-performance) web framework for building APIs with Python 3.7+ based on standard Python type hints. It automatically provides data validation, serialization, and interactive API documentation (using OpenAPI and JSON Schema) out of the box, significantly speeding up development.


In [7]:
from pydantic import BaseModel
from typing import List


class RequirementAnalysis(BaseModel):
    problem: str
    expected_behavior: str
    affected_component: str
    technical_requirements: List[str]
    acceptance_criteria: List[str]

In [8]:
requirements_agent = Agent(
    name="Requirements Analysis Agent",

    instructions="""
    You are a senior software requirements analyst.

    Analyze the software problem given by the user.

    Determine:
    - the main problem
    - expected behavior
    - affected software component
    - technical requirements
    - acceptance criteria

    Do not write implementation code.
    """,

    model=gemini_model,

    output_type=RequirementAnalysis
)

In [9]:
problem = """
My FastAPI login endpoint crashes when
the password field is empty.
"""

result = await Runner.run(
    requirements_agent,
    problem
)

print(result.final_output)

problem='The FastAPI login endpoint crashes when the password field is empty, leading to an unhandled exception.' expected_behavior='The FastAPI login endpoint should gracefully handle an empty password field by returning an appropriate HTTP error response (e.g., 400 Bad Request) with a clear error message, instead of crashing.' affected_component='FastAPI login endpoint and its associated input validation logic.' technical_requirements=["Implement input validation for the 'password' field in the FastAPI login endpoint, ensuring it is not empty or null.", 'Configure the validation to return an HTTP 400 Bad Request status code when the password field is invalid (empty or missing).', 'Provide a clear and descriptive error message in the response body indicating that the password field is required or cannot be empty.', 'Ensure the application does not raise unhandled exceptions when processing requests with empty or missing password fields.', "Update the endpoint's OpenAPI schema to refle

In [10]:
analysis = result.final_output

print("PROBLEM:")
print(analysis.problem)

print("\nEXPECTED BEHAVIOR:")
print(analysis.expected_behavior)

print("\nAFFECTED COMPONENT:")
print(analysis.affected_component)

print("\nTECHNICAL REQUIREMENTS:")

for requirement in analysis.technical_requirements:
    print("-", requirement)

print("\nACCEPTANCE CRITERIA:")

for criteria in analysis.acceptance_criteria:
    print("-", criteria)

PROBLEM:
The FastAPI login endpoint crashes when the password field is empty, leading to an unhandled exception.

EXPECTED BEHAVIOR:
The FastAPI login endpoint should gracefully handle an empty password field by returning an appropriate HTTP error response (e.g., 400 Bad Request) with a clear error message, instead of crashing.

AFFECTED COMPONENT:
FastAPI login endpoint and its associated input validation logic.

TECHNICAL REQUIREMENTS:
- Implement input validation for the 'password' field in the FastAPI login endpoint, ensuring it is not empty or null.
- Configure the validation to return an HTTP 400 Bad Request status code when the password field is invalid (empty or missing).
- Provide a clear and descriptive error message in the response body indicating that the password field is required or cannot be empty.
- Ensure the application does not raise unhandled exceptions when processing requests with empty or missing password fields.
- Update the endpoint's OpenAPI schema to reflect 

In [11]:
class CodeSolution(BaseModel):
    explanation: str
    code: str
    files_affected: List[str]

In [12]:
coding_agent = Agent(
    name="Coding Assistant",

    instructions="""
    You are a senior Python software engineer.

    Receive software requirements and propose
    an implementation.

    Produce:
    - explanation of the solution
    - corrected/generated code
    - files that may need modification

    Write clean and secure code.
    """,

    model=gemini_model,

    output_type=CodeSolution
)

In [13]:
coding_result = await Runner.run(
    coding_agent,
    str(analysis.model_dump())
)

solution = coding_result.final_output

print("EXPLANATION:")
print(solution.explanation)

print("\nCODE:")
print(solution.code)

print("\nFILES:")
print(solution.files_affected)

EXPLANATION:
The issue of the FastAPI login endpoint crashing with an empty password field is addressed by implementing robust input validation using Pydantic and a custom exception handler. The solution involves defining a Pydantic model for the login request body, specifying that the `password` field is required and must have a minimum length of 1 character. FastAPI automatically leverages Pydantic for validation, raising a `RequestValidationError` when validation fails.

To meet the requirement of returning an `HTTP 400 Bad Request` (instead of FastAPI's default `HTTP 422 Unprocessable Entity` for validation errors) and providing clearer error messages, a custom exception handler for `RequestValidationError` is implemented. This handler intercepts validation errors, specifically targeting cases where the `password` is empty or missing, and transforms them into a `400 Bad Request` response with descriptive error messages like 'Password cannot be empty' or 'Password is required'. This

In [14]:
class ReviewResult(BaseModel):
    approved: bool
    issues: List[str]
    suggestions: List[str]

In [15]:
reviewer_agent = Agent(
    name="Code Reviewer",

    instructions="""
    You are a senior software code reviewer.

    Review code for:

    - correctness
    - bugs
    - security issues
    - readability
    - maintainability
    - edge cases

    Decide whether the solution should be approved.
    """,

    model=gemini_model,

    output_type=ReviewResult
)

In [16]:
review_result = await Runner.run(
    reviewer_agent,
    solution.code
)

review = review_result.final_output

print("APPROVED:", review.approved)

print("\nISSUES:")
for issue in review.issues:
    print("-", issue)

print("\nSUGGESTIONS:")
for suggestion in review.suggestions:
    print("-", suggestion)

APPROVED: True

ISSUES:
- The login endpoint uses hardcoded username and password (`validuser`, `securepassword`) for authentication. While the code includes a comment stating this is for simulation, in a real-world application, hardcoding credentials is a significant security vulnerability. Passwords should be securely stored (hashed, not plaintext) and verified against a database.
- The custom error handling specifically targets password validation errors. If other fields require similar custom validation messages in the future, the exception handler might grow in complexity, becoming less modular. Consider a more generic approach for mapping common validation error types to custom messages.

SUGGESTIONS:
- For production applications, implement proper password hashing (e.g., using `bcrypt` or `argon2`) and secure storage practices for user credentials.
- For improved maintainability and testability in larger applications, consider extracting authentication logic into a separate depe

In [17]:
from pydantic import BaseModel
from typing import List

class TestingResult(BaseModel):
    test_cases: List[str]
    potential_failures: List[str]
    all_tests_passed: bool
    testing_summary: str

In [18]:
testing_agent = Agent(
    name="Testing Agent",

    instructions="""
    You are a senior software testing engineer.

    Analyze the provided source code and code review.

    Your responsibilities are:
    1. Generate important test cases.
    2. Check normal cases.
    3. Check edge cases.
    4. Check invalid inputs.
    5. Identify potential failures.
    6. Determine whether the code is likely to pass the tests.

    Do not modify the source code.
    """,

    model=gemini_model,
    output_type=TestingResult
)

In [19]:
testing_input = f"""
SOURCE CODE:

{solution.code}


CODE REVIEW:

Approved: {review.approved}

Issues:
{review.issues}

Suggestions:
{review.suggestions}
"""

testing_result = await Runner.run(
    testing_agent,
    testing_input
)

test = testing_result.final_output

In [20]:
print("TEST CASES:")

for case in test.test_cases:
    print("-", case)

print("\nPOTENTIAL FAILURES:")

for failure in test.potential_failures:
    print("-", failure)

print("\nALL TESTS PASSED:")
print(test.all_tests_passed)

print("\nSUMMARY:")
print(test.testing_summary)

TEST CASES:
- Test_1: Successful login with valid username 'validuser' and password 'securepassword'. Expect HTTP 200 with 'Login successful' message and token.
- Test_2: Access root endpoint. Expect HTTP 200 with 'Welcome to the API!' message.
- Test_3: Attempt login with missing username. Expect HTTP 400 and Pydantic validation error for missing username.
- Test_4: Attempt login with missing password. Expect HTTP 400 and custom validation error 'Password is required' for password field.
- Test_5: Attempt login with an empty string password (''). Expect HTTP 400 and custom validation error 'Password cannot be empty' for password field.
- Test_6: Attempt login with a password containing only a single space (' '). This passes Pydantic's 'min_length=1'. Expect HTTP 401 'Invalid credentials' as it doesn't match 'securepassword'.
- Test_7: Attempt login with correct username 'validuser' but incorrect password 'wrongpassword'. Expect HTTP 401 'Invalid credentials'.
- Test_8: Attempt login w

In [21]:
class BugAnalysis(BaseModel):
    bugs_found: List[str]
    root_causes: List[str]
    severity: str
    recommended_fixes: List[str]

In [22]:
bug_agent = Agent(
    name="Bug Investigation Agent",

    instructions="""
    You are a senior software debugging engineer.

    Analyze the source code, code review, and testing results.

    Your responsibilities:
    1. Identify bugs and vulnerabilities.
    2. Determine the root cause of each important problem.
    3. Determine overall severity.
    4. Recommend specific fixes.

    Do not rewrite the entire application.
    Focus on diagnosing the problems.
    """,

    model=gemini_model,
    output_type=BugAnalysis
)

In [23]:
bug_input = f"""
SOURCE CODE:
{solution.code}

CODE REVIEW ISSUES:
{review.issues}

TESTING FAILURES:
{test.potential_failures}
"""

bug_result = await Runner.run(
    bug_agent,
    bug_input
)

bug = bug_result.final_output

In [24]:
print("BUGS FOUND:")

for item in bug.bugs_found:
    print("-", item)

print("\nROOT CAUSES:")

for cause in bug.root_causes:
    print("-", cause)

print("\nSEVERITY:")
print(bug.severity)

print("\nRECOMMENDED FIXES:")

for fix in bug.recommended_fixes:
    print("-", fix)

BUGS FOUND:
- Hardcoded credentials in the /login endpoint (`validuser`, `securepassword`) pose a critical security vulnerability.
- The password validation `min_length=1` allows passwords consisting solely of whitespace (e.g., ' '). This bypasses the intent of 'Password cannot be empty' and creates insecure accounts.
- The custom exception handler relies on internal Pydantic error types (`string_too_short`, `missing`) and specific `loc` tuple structure `("body", "password")`. This makes the handler brittle and susceptible to breaking if Pydantic's internal error representation changes in future versions.
- Custom error messages are hardcoded directly within the `validation_exception_handler`. This reduces maintainability and scalability, as any new custom messages or changes for different fields would require modifying the handler directly, increasing its complexity.
- The custom message 'Password cannot be empty' is only triggered for an empty string (`''`), not for strings containin

In [28]:
fix_input = f"""
You previously generated this code:

{solution.code}

The Bug Investigation Agent found these bugs:

{bug.bugs_found}

Root causes:

{bug.root_causes}

Recommended fixes:

{bug.recommended_fixes}

Create an improved Version 2 of the code.

Fix the identified problems while preserving the original requirements.
"""

In [29]:
fixed_result = await Runner.run(
    coding_agent,
    fix_input
)

fixed_solution = fixed_result.final_output

In [30]:
print("IMPROVED CODE - VERSION 2")
print("=" * 60)

print("\nEXPLANATION:")
print(fixed_solution.explanation)

print("\nCODE:")
print(fixed_solution.code)

print("\nFILES AFFECTED:")
for file in fixed_solution.files_affected:
    print("-", file)

IMPROVED CODE - VERSION 2

EXPLANATION:
This updated version of the FastAPI application addresses all the identified bugs, enhancing security, robustness, and maintainability while preserving the original functionality.The key improvements are:

1.  **Secure Authentication Simulation (Hardcoded Credentials Fix):**
    *   Instead of hardcoded `validuser` and `securepassword`, the application now simulates a user database (`fake_users_db`) where passwords are stored as `bcrypt` hashes. This is crucial for security, as plaintext passwords are never stored. A `hash_password` function is used to hash passwords when 'creating' users, and `check_password` is used to securely verify user-provided passwords during login.

2.  **Robust Password Validation (Whitespace & Empty Fix):**
    *   The `UserLogin` Pydantic model's `password` field now includes `strip_whitespace=True`. This automatically removes leading/trailing whitespace from the input password before further validation.
    *   A cus

In [31]:
second_review_result = await Runner.run(
    reviewer_agent,
    fixed_solution.code
)

second_review = second_review_result.final_output

print("SECOND REVIEW")
print("=" * 50)

print("APPROVED:", second_review.approved)

print("\nISSUES:")
for issue in second_review.issues:
    print("-", issue)

print("\nSUGGESTIONS:")
for suggestion in second_review.suggestions:
    print("-", suggestion)

SECOND REVIEW
APPROVED: True

ISSUES:

SUGGESTIONS:
- Consider adding `strip_whitespace=True` and a `min_length` to the `username` field in `UserLogin` to ensure it's not empty or just whitespace, similar to the password field, for consistent input validation.
- For a production application, implement rate limiting on the `/login` endpoint to protect against brute-force attacks.
- The current `token: "fake-jwt-token"` is a placeholder. In a real application, a secure JSON Web Token (JWT) or similar authentication token should be generated and returned upon successful login.
- While the current custom exception handler is effective, for larger applications or more complex validation scenarios, you might consider defining custom exception classes (e.g., `InvalidPasswordError`) instead of relying solely on `ValueError` and string matching `error_type`. This can improve type-safety and make the handler logic slightly cleaner.


In [32]:
class DocumentationResult(BaseModel):
    project_summary: str
    changes_made: List[str]
    setup_instructions: List[str]
    usage_instructions: List[str]

In [33]:
documentation_agent = Agent(
    name="Documentation Agent",

    instructions="""
    You are a senior technical documentation writer.

    Analyze the final software solution.

    Generate clear project documentation containing:

    1. Project summary
    2. Changes made
    3. Setup instructions
    4. Usage instructions

    Write documentation suitable for a GitHub README.
    """,

    model=gemini_model,
    output_type=DocumentationResult
)

In [34]:
doc_result = await Runner.run(
    documentation_agent,
    fixed_solution.code
)

documentation = doc_result.final_output

In [35]:
print("PROJECT SUMMARY")
print(documentation.project_summary)

print("\nCHANGES MADE")
for change in documentation.changes_made:
    print("-", change)

print("\nSETUP INSTRUCTIONS")
for step in documentation.setup_instructions:
    print("-", step)

print("\nUSAGE INSTRUCTIONS")
for step in documentation.usage_instructions:
    print("-", step)

PROJECT SUMMARY
This project implements a secure user login API using FastAPI, focusing on robust input validation, centralized error handling, and secure password management. It addresses common vulnerabilities and best practices by integrating bcrypt for password hashing and providing detailed, user-friendly error messages for validation failures.

CHANGES MADE
- **Centralized Error Messages**: Custom error messages are now defined in a `ERROR_MESSAGES` dictionary, improving consistency and maintainability, fixing hardcoded custom error messages.
- **Enhanced Password Validation**: The `UserLogin` Pydantic model now includes `min_length=1` and `strip_whitespace=True` for the password field. A custom validator (`password_not_whitespace_only`) further ensures that the password is not empty or consists solely of whitespace after stripping, addressing insufficient password validation.
- **Custom Exception Handling**: A dedicated exception handler for `RequestValidationError` has been imp

In [36]:
from agents import function_tool
import os

In [37]:
@function_tool
def read_file(file_path: str) -> str:
    """
    Read the contents of a project file.
    """
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            return file.read()
    except Exception as e:
        return f"Error reading file: {str(e)}"


In [38]:
@function_tool
def write_file(file_path: str, content: str) -> str:
    """
    Write content to a project file.
    """
    try:
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(content)

        return f"Successfully wrote to {file_path}"

    except Exception as e:
        return f"Error writing file: {str(e)}"

In [39]:
@function_tool
def search_code(file_path: str, search_term: str) -> str:
    """
    Search for a term inside a source code file.
    """
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            lines = file.readlines()

        matches = []

        for number, line in enumerate(lines, start=1):
            if search_term.lower() in line.lower():
                matches.append(
                    f"Line {number}: {line.strip()}"
                )

        if not matches:
            return "No matches found."

        return "\n".join(matches)

    except Exception as e:
        return f"Error searching code: {str(e)}"


In [40]:
sample_code = """
def login(username, password):
    if username == "admin" and password == "1234":
        return "Login successful"

    return "Invalid credentials"
"""

with open("sample_app.py", "w") as file:
    file.write(sample_code)

print("sample_app.py created!")

sample_app.py created!


In [41]:
file_agent = Agent(
    name="Code File Analyst",

    instructions="""
    You analyze software project files.

    Use the available tools when necessary.

    When the user asks you to inspect a file:
    1. Read the file.
    2. Search relevant code when useful.
    3. Explain any problems you find.

    Do not invent file contents.
    """,

    model=gemini_model,

    tools=[
        read_file,
        search_code
    ]
)

In [42]:
file_result = await Runner.run(
    file_agent,
    """
    Read sample_app.py and analyze its login function.
    Identify any security problems.
    """
)

print(file_result.final_output)


The `login` function in `sample_app.py` has significant security problems:

*   **Hardcoded Credentials**: The username "admin" and password "1234" are hardcoded directly into the function. This makes the application highly vulnerable as these credentials are easy to discover by anyone with access to the code.
*   **No Password Hashing**: Passwords should never be stored or compared in plain text. They should always be hashed using a strong, one-way hashing algorithm (like bcrypt or scrypt) to protect against credential exposure if the system is breached.
*   **Weak Password**: The password "1234" is extremely weak and easily guessable, making the application susceptible to brute-force attacks.
*   **No User Management**: There's no mechanism to add new users, change passwords, or handle different user roles.
*   **No Brute-Force Protection**: The function lacks any measures to prevent or mitigate brute-force attacks, such as rate limiting or account lockouts after multiple failed logi

In [43]:
!pip install -q pytest


In [44]:
test_code = """
from sample_app import login

def test_valid_login():
    assert login("admin", "1234") == "Login successful"

def test_wrong_password():
    assert login("admin", "wrong") == "Invalid credentials"

def test_wrong_username():
    assert login("user", "1234") == "Invalid credentials"

def test_empty_password():
    assert login("admin", "") == "Invalid credentials"
"""

with open("test_sample_app.py", "w") as file:
    file.write(test_code)

print("Test file created!")

Test file created!


In [45]:
import subprocess

@function_tool
def run_tests(test_file: str) -> str:
    """
    Run pytest on a Python test file and return the test results.
    """
    try:
        result = subprocess.run(
            ["python", "-m", "pytest", test_file, "-q"],
            capture_output=True,
            text=True,
            timeout=30
        )

        output = result.stdout + "\n" + result.stderr

        return f"""
Exit Code: {result.returncode}

Test Output:
{output}
"""

    except Exception as e:
        return f"Error running tests: {str(e)}"

In [46]:
testing_agent_with_tools = Agent(
    name="Testing Agent",

    instructions="""
    You are a software testing engineer.

    You have access to a real pytest execution tool.

    When asked to run tests:
    1. Use run_tests.
    2. Analyze the actual pytest output.
    3. Clearly report which tests passed or failed.
    4. Explain failures if there are any.

    Never claim that a test passed unless the tool output
    confirms it.
    """,

    model=gemini_model,

    tools=[
        run_tests
    ]
)

In [47]:
actual_test_result = await Runner.run(
    testing_agent_with_tools,
    """
    Run the tests in test_sample_app.py.

    Tell me:
    - how many tests passed
    - how many failed
    - whether the application passed testing
    """
)

print(actual_test_result.final_output)

All 4 tests in `test_sample_app.py` passed.
0 tests failed.
The application passed testing.


In [ ]:
!pip install -q requests

In [48]:
import requests

@function_tool
def get_github_issue(
    owner: str,
    repo: str,
    issue_number: int
) -> str:
    """
    Fetch a public GitHub issue using the GitHub API.
    """

    url = (
        f"https://api.github.com/repos/"
        f"{owner}/{repo}/issues/{issue_number}"
    )

    try:
        response = requests.get(url, timeout=15)

        if response.status_code != 200:
            return (
                f"GitHub API error: "
                f"{response.status_code} - {response.text}"
            )

        data = response.json()

        return f"""
Issue Number: {data['number']}
Title: {data['title']}
State: {data['state']}

Description:
{data.get('body', 'No description provided')}
"""

    except Exception as e:
        return f"Error fetching GitHub issue: {str(e)}"

In [49]:
github_requirements_agent = Agent(
    name="GitHub Requirements Analysis Agent",

    instructions="""
    You are a senior software requirements analyst.

    When the user provides a GitHub repository and issue number:

    1. Use the get_github_issue tool.
    2. Read the actual GitHub issue.
    3. Identify the software problem.
    4. Explain the expected behavior.
    5. Identify likely affected components.
    6. Extract technical requirements.

    Always use the GitHub tool when repository issue
    information is provided.

    Do not invent issue contents.
    """,

    model=gemini_model,

    tools=[
        get_github_issue
    ]
)

In [50]:
github_result = await Runner.run(
    github_requirements_agent,
    """
    Analyze GitHub issue #1 from:

    Owner: pallets
    Repository: flask
    """
)

print(github_result.final_output)

**Software Problem:** The current Flask URL endpoint system relies solely on function names, which can lead to naming conflicts and poor organization in large applications composed of multiple modules or components.

**Expected Behavior:** The system should allow developers to use "dotted names" for URL endpoints. This means the module name would be incorporated into the endpoint name, providing a more structured and unique identifier for routes, especially in larger applications.

**Likely Affected Components:**
*   **URL Routing System:** The core component responsible for mapping URLs to view functions will need modification to interpret and resolve dotted names.
*   **Endpoint Registration:** The mechanism by which view functions and their associated endpoints are registered within the Flask application will need to be updated to accept and process dotted names.
*   **Application Configuration:** While not explicitly stated, there might be a need for configuration options related t

In [79]:
handoff_coding_agent = Agent(
    name="Handoff Coding Agent",

    instructions="""
    You are a senior Python developer.

    You receive analyzed software requirements.

    Create a proposed implementation and explain
    the code changes required.

    Return the solution to the user.
    """,

    model=gemini_model
)

In [80]:
handoff_requirements_agent = Agent(
    name="Handoff Requirements Agent",

    instructions="""
    You are a senior software requirements analyst.

    First analyze the user's software problem.

    Identify:
    - problem
    - expected behavior
    - affected component
    - technical requirements

    After understanding the requirements,
    hand the task to the Handoff Coding Agent
    so that it can propose the implementation.
    """,

    model=gemini_model,

    handoffs=[
        handoff_coding_agent
    ]
)

In [73]:
from agents.memory import OpenAIConversationsSession

# Initialize OpenAIConversationsSession to maintain conversational memory
chat_history = OpenAIConversationsSession()


In [66]:
import openai

try:
    # Attempt to initialize AsyncOpenAI directly
    test_openai_client = openai.AsyncOpenAI()
    print("AsyncOpenAI client initialized successfully.")
except openai.OpenAIError as e:
    print(f"Failed to initialize AsyncOpenAI client: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

AsyncOpenAI client initialized successfully.


In [86]:
handoff_result = await Runner.run(
    handoff_requirements_agent,
    """
    My FastAPI login endpoint crashes when
    the password field is empty.

    Analyze the problem and create a solution.
    """
)

print(handoff_result.final_output)


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



RateLimitError: litellm.RateLimitError: litellm.RateLimitError: geminiException - {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 49.670546607s.",
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.Help",
        "links": [
          {
            "description": "Learn more about Gemini API quotas",
            "url": "https://ai.google.dev/gemini-api/docs/rate-limits"
          }
        ]
      },
      {
        "@type": "type.googleapis.com/google.rpc.QuotaFailure",
        "violations": [
          {
            "quotaMetric": "generati

In [82]:
print(help(Agent))
print(help(Runner))

Help on class Agent in module agents.agent:

class Agent(AgentBase, typing.Generic)
 |  Agent(name: 'str', handoff_description: 'str | None' = None, tools: 'list[Tool]' = <factory>, mcp_servers: 'list[MCPServer]' = <factory>, mcp_config: 'MCPConfig' = <factory>, instructions: 'str | Callable[[RunContextWrapper[TContext], Agent[TContext]], MaybeAwaitable[str]] | None' = None, prompt: 'Prompt | DynamicPromptFunction | None' = None, handoffs: 'list[Agent[Any] | Handoff[TContext, Any]]' = <factory>, model: 'str | Model | None' = None, model_settings: 'ModelSettings' = <factory>, input_guardrails: 'list[InputGuardrail[TContext]]' = <factory>, output_guardrails: 'list[OutputGuardrail[TContext]]' = <factory>, output_type: 'type[Any] | AgentOutputSchemaBase | None' = None, hooks: 'AgentHooks[TContext] | None' = None, tool_use_behavior: "Literal['run_llm_again', 'stop_on_first_tool'] | StopAtTools | ToolsToFinalOutputFunction" = 'run_llm_again', reset_tool_choice: 'bool' = True) -> None
 |
 |  

In [54]:
print("Final Agent:")
print(handoff_result.last_agent.name)

Final Agent:
Handoff Coding Agent


In [70]:
from agents.memory import Session

In [65]:
import os
from google.colab import userdata

# Set OpenAI API key for OpenAIConversationsSession if available
if not os.getenv("OPENAI_API_KEY"): # Check if already set
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        print("OPENAI_API_KEY loaded from Colab Secrets.")
    except userdata.exceptions.KeyNotFoundError:
        print("OpenAI API Key not found in Colab Secrets. Please add 'OPENAI_API_KEY' if you wish to use OpenAI models for conversational memory.")
    except Exception as e:
        print(f"An error occurred while loading OpenAI API Key: {e}")
else:
    print("OPENAI_API_KEY is already set.")

OPENAI_API_KEY loaded from Colab Secrets.
